# Social Media Crisis Detection & Response Monitor

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Analyze sentiment** in real-time using `AI_SENTIMENT()`
2. **Detect adverse events** with `AI_COMPLETE()` and LLM classification
3. **Calculate statistical baselines** using window functions and aggregations
4. **Identify anomalies** through z-score calculations
5. **Build alerting logic** for communications teams
6. **Create real-time dashboards** with Streamlit

---

## What You'll Build

A **crisis detection system** that monitors social media for brand threats:
- Real-time sentiment scoring across platforms (Twitter, Facebook, Reddit, TikTok)
- Adverse event signal detection using LLM
- Statistical anomaly detection (sentiment spikes, volume surges)
- Automated severity classification (CRITICAL, HIGH, MEDIUM, LOW)
- Interactive dashboard for rapid response

---

## Technical Requirements

- **Snowflake account** with Cortex AI enabled
- **Approximately 12-15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features

- `AI_SENTIMENT()` - Sentiment analysis
- `AI_COMPLETE()` - LLM classification
- `STDDEV()`, `AVG()` - Statistical baselines
- Window functions - Rolling calculations
- `CASE` statements - Severity classification

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `CRISIS_DETECTION_DB`
- **Schema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

### Real-Time Crisis Detection

Social media crises can escalate in hours. This system provides:
1. **Continuous monitoring** of brand mentions
2. **Automated sentiment scoring** on every post
3. **Statistical baselines** to detect anomalies
4. **Instant alerts** when metrics spike

### Why This Matters

Traditional monitoring requires manual review of thousands of posts. Automated detection allows communications teams to:
- **Respond within minutes** instead of hours
- **Prioritize severe issues** automatically
- **Track adverse events** for patient safety

In [ ]:
-- Create crisis detection environment
CREATE DATABASE IF NOT EXISTS CRISIS_DETECTION_DB;
CREATE SCHEMA IF NOT EXISTS CRISIS_DETECTION_DB.PUBLIC;
USE SCHEMA CRISIS_DETECTION_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name,
    'Crisis Detection Environment Ready!' as status;

---

## Paso 2: Define Data Structure

### Tables

1. **`social_media_posts`** - Raw posts from multiple platforms
2. **`post_sentiment_analysis`** - Cortex AI sentiment scores
3. **`adverse_event_signals`** - LLM-detected medical incidents
4. **`sentiment_baselines`** - Historical averages for anomaly detection
5. **`crisis_alerts`** - Active crisis scenarios

### Sentiment Scoring

- **+0.6 to +1.0**: Altoly positive (testimonials, success stories)
- **+0.2 to +0.6**: Positive (satisfied users)
- **-0.2 to +0.2**: Neutral (factual statements)
- **-0.6 to -0.2**: Negative (complaints, issues)
- **-1.0 to -0.6**: Altoly negative (angry, potential crisis)

In [ ]:
-- Social media posts table
CREATE OR REPLACE TABLE social_media_posts (
    post_id VARCHAR(50) PRIMARY KEY,
    platform VARCHAR(50),          -- twitter, facebook, instagram, reddit, tiktok
    author_username VARCHAR(100),
    post_text TEXT,
    posted_timestamp TIMESTAMP,
    engagement_count INTEGER,      -- likes + shares + comments
    product_mentioned VARCHAR(100),
    hashtags ARRAY,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Sentiment analysis results
CREATE OR REPLACE TABLE post_sentiment_analysis (
    post_id VARCHAR(50) PRIMARY KEY,
    sentiment_score FLOAT,                    -- -1.0 to +1.0
    sentiment_category VARCHAR(20),           -- POSITIVE, NEUTRAL, NEGATIVE
    adverse_event_detected BOOLEAN,           -- TRUE if medical incident mentioned
    adverse_event_type VARCHAR(500),          -- allergic_reaction, hospitalization, etc (LLM can be very verbose!)
    severity VARCHAR(20),                     -- CRITICAL, HIGH, MEDIUM, LOW
    analyzed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Sentiment baselines for anomaly detection
CREATE OR REPLACE TABLE sentiment_baselines (
    product VARCHAR(100),
    platform VARCHAR(50),
    baseline_date DATE,
    avg_daily_sentiment FLOAT,
    stddev_sentiment FLOAT,
    avg_daily_volume INTEGER,
    stddev_volume FLOAT,
    PRIMARY KEY (product, platform, baseline_date)
);

-- Crisis alerts
CREATE OR REPLACE TABLE crisis_alerts (
    alert_id VARCHAR(50) PRIMARY KEY,
    product VARCHAR(100),
    platform VARCHAR(50),
    alert_type VARCHAR(50),        -- sentiment_drop, volume_spike, adverse_events
    severity VARCHAR(20),
    current_sentiment FLOAT,
    z_score FLOAT,                 -- standard deviations from baseline
    post_count INTEGER,
    detected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    resolved BOOLEAN DEFAULT FALSE
);

SELECT 'Tables created successfully!' as status;

---

## Paso 3: Generar Datos Sintéticos Social Media Data

### Qué Vamos a Crear

- **1,000 social media posts** across 5 platforms
- **Mixed sentiment**: 50% positive, 30% neutral, 20% negative
- **Products**: Xarelto, Eylea, Stivarga (Bayer medications)
- **Last 7 days** of data with realistic timestamps
- **Include adverse events**: ~2% of posts mention medical incidents

In [ ]:
-- Generar realistic social media posts
INSERT INTO social_media_posts
WITH platforms AS (
    SELECT * FROM (VALUES
        ('twitter'), ('facebook'), ('instagram'), ('reddit'), ('tiktok')
    ) t(platform)
),
products AS (
    SELECT * FROM (VALUES 
        ('Xarelto'), ('Eylea'), ('Stivarga')
    ) t(product)
),
post_templates AS (
    SELECT * FROM (VALUES
        -- Altoly positive (0.8-1.0)
        ('Just hit my 6-month milestone with {product}! No stroke events and stable INR. Life changing medication! #afib #strokeprevention', 0.9),
        ('{product} has completely transformed my health. My doctor says I''m a completely different patient. Forever grateful! #health', 0.85),
        ('Best decision ever to start {product}. Energy is back, sugar cravings gone, and my labs are perfect. Thank you!', 0.8),
        
        -- Positive (0.3-0.7)
        ('3 months on {product}. Blood sugar is stable and I''ve lost 15 pounds. The nausea was rough at first but worth it.', 0.6),
        ('Happy with my {product} results so far. A1C dropped 1.5 points. Still adjusting to the weekly injection schedule.', 0.5),
        ('{product} is working well for me. No major side effects and steady progress. My endo is pleased.', 0.4),
        
        -- Neutral (0-0.2)
        ('Week 4 of {product}. Some nausea and tiredness. Blood sugar slightly better. Hoping it improves.', 0.1),
        ('Started {product} last month. Too early to see major changes but following my doctor''s guidance.', 0.05),
        ('{product} update: manageable side effects, modest weight loss. Sticking with it per my treatment plan.', 0),
        
        -- Negative (-0.3 to -0.7)
        ('The nausea from {product} is unbearable. Week 5 and still can''t keep food down. Calling my doctor tomorrow.', -0.5),
        ('{product} is SO expensive. $900/month even with insurance. How is anyone supposed to afford this?', -0.4),
        ('Severe constipation and stomach pain on {product}. This is miserable. Not sure I can continue.', -0.6),
        
        -- Altoly negative (-0.8 to -1.0)  
        ('Had to stop {product} immediately. Severe allergic reaction - hives, throat swelling. In ER now.', -0.9),
        ('{product} caused pancreatitis. Hospitalized for 4 days. This drug is dangerous.', -0.95),
        ('DANGER: {product} made my gallbladder fail. Emergency surgery. Lawsuit pending.', -1.0)
    ) t(template, expected_sentiment)
)
SELECT 
    'POST_' || LPAD(SEQ4()::VARCHAR, 8, '0') as post_id,
    p.platform,
    'user_' || LPAD(UNIFORM(1, 99999, RANDOM())::VARCHAR, 5, '0') as author_username,
    REPLACE(pt.template, '{product}', pr.product) as post_text,
    DATEADD('minute', -FLOOR(UNIFORM(1, 10080, RANDOM())), CURRENT_TIMESTAMP()) as posted_timestamp,
    FLOOR(UNIFORM(5, 5000, RANDOM())) as engagement_count,
    pr.product as product_mentioned,
    ARRAY_CONSTRUCT('#afib', '#strokeprevention', '#health') as hashtags,
    CURRENT_TIMESTAMP() as ingested_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000)) g
CROSS JOIN platforms p
CROSS JOIN products pr
CROSS JOIN post_templates pt
WHERE UNIFORM(0, 1, RANDOM()) < 0.15  -- Random sampling
QUALIFY ROW_NUMBER() OVER (ORDER BY UNIFORM(0, 1, RANDOM())) <= 1000;

SELECT 
    COUNT(*) as total_posts,
    COUNT(DISTINCT platform) as platforms,
    COUNT(DISTINCT product_mentioned) as products,
    MIN(posted_timestamp) as earliest_post,
    MAX(posted_timestamp) as latest_post
FROM social_media_posts;

---

## Paso 4: Analyze Sentiment with Cortex AI

### The `SENTIMENT()` Function

```sql
AI_SENTIMENT(text_column)
```

**Cómo funciona:**
- Returns a float between -1.0 (very negative) and +1.0 (very positive)
- Pre-trained on billions of text samples
- Understands context, sarcasm, medical terminology
- No training or model configuration required

### Classification Logic

We'll categorize sentiment into buckets:
- **POSITIVE**: score >= 0.3
- **NEUTRAL**: score between -0.2 and 0.3
- **NEGATIVE**: score < -0.2

In [ ]:
-- Analyze sentiment for all posts using Cortex AI
-- Verified via Snowflake docs 2025-01: AI_SENTIMENT returns OBJECT
INSERT INTO post_sentiment_analysis (
    post_id, 
    sentiment_score, 
    sentiment_category,
    adverse_event_detected,
    severity
)
SELECT 
    post_id,
    -- Extract numeric sentiment score (Use Case 28 pattern)
    CASE AI_SENTIMENT(post_text)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 1.0
        WHEN 'neutral' THEN 0.0
        WHEN 'negative' THEN -1.0
        WHEN 'mixed' THEN -0.5
        ELSE -1.0
    END as sentiment_score,
    -- Human-readable sentiment category
    CASE AI_SENTIMENT(post_text)['categories'][0]['sentiment']::STRING
        WHEN 'positive' THEN 'Positive'
        WHEN 'neutral' THEN 'Neutral'
        WHEN 'negative' THEN 'Negative'
        WHEN 'mixed' THEN 'Negative'
        ELSE 'Very Negative'
    END as sentiment_category,
    -- Detect potential adverse events based on keywords
    CASE
        WHEN LOWER(post_text) LIKE '%er%' 
          OR LOWER(post_text) LIKE '%emergency%'
          OR LOWER(post_text) LIKE '%hospital%'
          OR LOWER(post_text) LIKE '%allergic%'
          OR LOWER(post_text) LIKE '%severe%'
          OR LOWER(post_text) LIKE '%pancreatitis%'
        THEN TRUE
        ELSE FALSE
    END as adverse_event_detected,
    -- Severity based on sentiment + adverse events
    CASE 
        WHEN CASE
            WHEN LOWER(post_text) LIKE '%er%' 
              OR LOWER(post_text) LIKE '%emergency%'
              OR LOWER(post_text) LIKE '%hospital%'
              OR LOWER(post_text) LIKE '%allergic%'
              OR LOWER(post_text) LIKE '%severe%'
              OR LOWER(post_text) LIKE '%pancreatitis%'
            THEN TRUE
            ELSE FALSE
        END = TRUE 
        AND AI_SENTIMENT(post_text)['categories'][0]['sentiment']::STRING IN ('negative', 'mixed')
        THEN 'CRITICAL'
        WHEN AI_SENTIMENT(post_text)['categories'][0]['sentiment']::STRING = 'negative' THEN 'HIGH'
        WHEN AI_SENTIMENT(post_text)['categories'][0]['sentiment']::STRING = 'mixed' THEN 'MEDIUM'
        WHEN AI_SENTIMENT(post_text)['categories'][0]['sentiment']::STRING = 'neutral' THEN 'LOW'
        ELSE 'LOW'
    END as severity
FROM social_media_posts;

-- View sentiment distribution
SELECT 
    sentiment_category,
    COUNT(*) as post_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as percentage,
    ROUND(AVG(sentiment_score), 3) as avg_score
FROM post_sentiment_analysis
GROUP BY sentiment_category
ORDER BY post_count DESC;

---

## Paso 5: Classify Adverse Events with LLM

### The `COMPLETE()` Function

```sql
AI_COMPLETE(model, prompt)
```

We'll use LLM to classify adverse event types:
- **allergic_reaction**: Hives, swelling, anaphylaxis
- **gastrointestinal**: Severe nausea, vomiting, pancreatitis
- **hospitalization**: ER visits, surgery
- **other_serious**: Any other serious medical event

In [ ]:
-- Use LLM to classify adverse event types for severe posts
-- Using MERGE to update with LLM classification results
MERGE INTO post_sentiment_analysis t
USING (
    SELECT 
        psa.post_id,
        -- Extract just the first word from LLM response (in case it's verbose)
        TRIM(SPLIT_PART(
            AI_COMPLETE(
                'mistral-large',
                'You are a medical classifier. Output ONLY ONE WORD from this list: allergic_reaction, gastrointestinal, hypoglycemia, injection_site, other_serious, not_adverse_event. Post: ' || p.post_text || ' Answer (one word only):'
            ), ' ', 1
        )) as classified_type
    FROM post_sentiment_analysis psa
    INNER JOIN social_media_posts p ON psa.post_id = p.post_id
    WHERE psa.adverse_event_detected = TRUE
    LIMIT 10  -- Limit to 10 for demo (LLM calls can be slow)
) s
ON t.post_id = s.post_id
WHEN MATCHED THEN UPDATE SET t.adverse_event_type = s.classified_type;

-- View classified adverse events
SELECT 
    adverse_event_type,
    COUNT(*) as event_count,
    ROUND(AVG(sentiment_score), 3) as avg_sentiment
FROM post_sentiment_analysis
WHERE adverse_event_type IS NOT NULL
GROUP BY adverse_event_type
ORDER BY event_count DESC;

---

## Paso 6: Calculate Statistical Baselines

### Qué Vamos a Crear

Statistical baselines for **normal** social media activity:
- **Mean sentiment** per day, product, and platform  
- **Standard deviation** of sentiment (how much it varies)
- **Average daily volume** (typical post count)
- **Rolling historical data** to capture patterns

### Why Baselines Matter

Without historical context, you can't distinguish normal fluctuations from true crises:
- **Negative sentiment** might be normal for a product category (anticoagulants often have mixed sentiment due to bleeding concerns)
- **Volume spikes** could be seasonal (New Year's resolutions) or campaign-driven (new ads)
- **Baselines** provide the statistical foundation for anomaly detection

**Real-world example:** If a product normally has sentiment of -0.2 (slightly negative), a drop to -0.4 isn't alarming. But if normal is +0.3 (positive), that same -0.4 signals a crisis.

### The Z-Score Approach

We use **z-scores** to detect anomalies:

```sql
z-score = (current_value - baseline_mean) / baseline_stddev
```

**Interpretation:**
- z = 0: Exactly average behavior
- z = -2: Two standard deviations below average (warning threshold)
- z = -3: Three standard deviations below average (crisis threshold)  
- z < -2 occurs only ~2.5% of the time in normal distributions

This is the same statistical method used in:
- **Financial markets** for fraud detection
- **Medical diagnostics** for identifying outliers
- **Manufacturing** for quality control

In [ ]:
-- Calculate daily sentiment baselines for each product/platform
INSERT INTO sentiment_baselines
SELECT 
    p.product_mentioned as product,
    p.platform,
    DATE_TRUNC('day', p.posted_timestamp)::DATE as baseline_date,
    ROUND(AVG(s.sentiment_score), 4) as avg_daily_sentiment,
    ROUND(STDDEV(s.sentiment_score), 4) as stddev_sentiment,
    COUNT(*) as avg_daily_volume,
    ROUND(STDDEV(COUNT(*)) OVER (
        PARTITION BY p.product_mentioned, p.platform
    ), 2) as stddev_volume
FROM social_media_posts p
INNER JOIN post_sentiment_analysis s ON p.post_id = s.post_id
GROUP BY 
    p.product_mentioned, 
    p.platform, 
    DATE_TRUNC('day', p.posted_timestamp)::DATE;

-- View baselines
SELECT 
    product,
    platform,
    COUNT(*) as days_tracked,
    ROUND(AVG(avg_daily_sentiment), 3) as overall_avg_sentiment,
    ROUND(AVG(stddev_sentiment), 3) as avg_stddev,
    ROUND(AVG(avg_daily_volume), 0) as avg_daily_posts
FROM sentiment_baselines
GROUP BY product, platform
ORDER BY product, platform;

---

## Paso 7: Detect Crisis Scenarios

### What We're Building

A **real-time crisis detection system** that triggers alerts when:
1. **Sentiment drops significantly** (z-score < -2.0)
2. **Volume spikes** (>150% of baseline)
3. **Multiple adverse events** reported (>3 in 6 hours)

### The CTE Pattern

This query uses **Common Table Expressions (CTEs)** to build logic step-by-step:

```sql
WITH recent_6hr AS (...),      -- Current 6-hour window
     historical_baseline AS (...) -- 7-day baseline
SELECT ...                      -- Compare current vs baseline
```

**Why CTEs?** 
- Breaks complex logic into readable steps
- Makes debugging easier (can test each CTE separately)
- Better performance than subqueries in many cases

### Alert Severity Classification

- **CRITICAL**: Z-score < -3.0 OR ≥5 adverse events (immediate escalation)
- **HIGH**: Z-score < -2.5 OR 3-4 adverse events (notify senior team)
- **MEDIUM**: Z-score < -2.0 OR ≥150% volume spike (monitor closely)
- **LOW**: Minor anomalies (track trends)

### Why 6-Hour Windows?

Social media crises unfold rapidly:
- **Hour 1-2**: Initial posts appear
- **Hour 3-4**: Pattern becomes detectable
- **Hour 5-6**: Requires response if trend continues

Longer windows (24hr) miss fast-moving issues. Shorter windows (1hr) have too much noise.

In [ ]:
-- Create crisis detection view with real-time calculations
CREATE OR REPLACE VIEW crisis_detection_realtime AS
WITH recent_6hr AS (
    -- Last 6 hours of data per product/platform
    SELECT 
        p.product_mentioned as product,
        p.platform,
        AVG(s.sentiment_score) as current_avg_sentiment,
        COUNT(*) as current_post_volume,
        SUM(CASE WHEN s.adverse_event_detected THEN 1 ELSE 0 END) as adverse_event_count
    FROM social_media_posts p
    INNER JOIN post_sentiment_analysis s ON p.post_id = s.post_id
    WHERE p.posted_timestamp >= DATEADD('hour', -6, CURRENT_TIMESTAMP())
    GROUP BY p.product_mentioned, p.platform
    HAVING COUNT(*) >= 3  -- Minimum posts for statistical significance
),
historical_baseline AS (
    -- Historical averages (last 7 days)
    SELECT 
        product,
        platform,
        AVG(avg_daily_sentiment) as baseline_sentiment,
        AVG(stddev_sentiment) as baseline_stddev,
        AVG(avg_daily_volume) as baseline_volume
    FROM sentiment_baselines
    WHERE baseline_date >= DATEADD('day', -7, CURRENT_DATE())
    GROUP BY product, platform
)
SELECT 
    r.product,
    r.platform,
    ROUND(r.current_avg_sentiment, 3) as current_sentiment,
    r.current_post_volume,
    r.adverse_event_count,
    ROUND(h.baseline_sentiment, 3) as baseline_sentiment,
    -- Calculate z-score
    ROUND(
        (r.current_avg_sentiment - h.baseline_sentiment) / NULLIF(h.baseline_stddev, 0),
        2
    ) as sentiment_z_score,
    -- Volume spike percentage
    ROUND(
        ((r.current_post_volume - h.baseline_volume) / NULLIF(h.baseline_volume, 0)) * 100,
        1
    ) as volume_spike_pct,
    -- Determine crisis severity
    CASE
        WHEN (r.current_avg_sentiment - h.baseline_sentiment) / NULLIF(h.baseline_stddev, 0) < -3.0 
          OR r.adverse_event_count >= 5
        THEN 'CRITICAL'
        WHEN (r.current_avg_sentiment - h.baseline_sentiment) / NULLIF(h.baseline_stddev, 0) < -2.5
          OR r.adverse_event_count >= 3
        THEN 'HIGH'
        WHEN (r.current_avg_sentiment - h.baseline_sentiment) / NULLIF(h.baseline_stddev, 0) < -2.0
        THEN 'MEDIUM'
        ELSE 'LOW'
    END as alert_severity,
    -- Crisis status
    CASE 
        WHEN (r.current_avg_sentiment - h.baseline_sentiment) / NULLIF(h.baseline_stddev, 0) < -2.0
          OR r.adverse_event_count >= 3
        THEN 'CRISIS_DETECTED'
        ELSE 'NORMAL'
    END as status
FROM recent_6hr r
INNER JOIN historical_baseline h 
    ON r.product = h.product 
    AND r.platform = h.platform
WHERE h.baseline_stddev IS NOT NULL;

-- View current crises
SELECT * 
FROM crisis_detection_realtime 
WHERE status = 'CRISIS_DETECTED'
ORDER BY sentiment_z_score;

---

## Paso 8: Interactive Crisis Monitoring Dashboard

### Dashboard Features

- **Real-time metrics**: Total posts, active crises, average sentiment
- **Crisis alerts**: Products and platforms with detected crises
- **Sentiment trends**: Distribution across positive/neutral/negative
- **Sample posts**: Recent negative posts requiring review
- **Adverse events**: Medical incidents flagged for pharmacovigilance

In [ ]:
# Crisis Detection Dashboard - Enhanced with Altair
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("🚨 Social Media Crisis Detection Dashboard")
st.markdown("### Real-Time Brand Monitoring & Response System")

# Create tabs for better organization
tab1, tab2, tab3, tab4 = st.tabs(["📊 Overview", "🚨 Active Crises", "📈 Trends", "🔍 Details"])

# ============================================================================
# TAB 1: OVERVIEW
# ============================================================================
with tab1:
    # Key metrics row with enhanced styling
    col1, col2, col3, col4 = st.columns(4)
    
    total_posts = session.sql("SELECT COUNT(*) as cnt FROM social_media_posts WHERE posted_timestamp >= DATEADD('hour', -24, CURRENT_TIMESTAMP())").collect()[0]['CNT']
    active_crises = session.sql("SELECT COUNT(*) as cnt FROM crisis_detection_realtime WHERE status = 'CRISIS_DETECTED'").collect()[0]['CNT']
    avg_sentiment = session.sql("SELECT ROUND(AVG(sentiment_score), 3) as avg FROM post_sentiment_analysis WHERE analyzed_at >= DATEADD('hour', -24, CURRENT_TIMESTAMP())").collect()[0]['AVG']
    adverse_events = session.sql("SELECT COUNT(*) as cnt FROM post_sentiment_analysis WHERE adverse_event_detected = TRUE AND analyzed_at >= DATEADD('hour', -24, CURRENT_TIMESTAMP())").collect()[0]['CNT']
    
    with col1:
        st.metric("24hr Posts", f"{total_posts:,}", help="Total posts analyzed in last 24 hours")
    
    with col2:
        crisis_status = "🚨 Alert" if active_crises > 0 else "✅ Clear"
        st.metric("Active Crises", active_crises, delta=crisis_status,
                  delta_color="inverse" if active_crises > 0 else "normal")
    
    with col3:
        sentiment_emoji = "😊" if avg_sentiment >= 0.5 else "😐" if avg_sentiment >= -0.5 else "😟"
        st.metric("Avg Sentiment", f"{avg_sentiment:.3f}", 
                  delta=f"{sentiment_emoji} {abs(avg_sentiment - 0.5):.3f}",
                  delta_color="normal" if avg_sentiment >= 0.5 else "inverse")
    
    with col4:
        st.metric("Adverse Events", adverse_events, 
                  delta="⚠️ High" if adverse_events > 5 else "✓ Normal",
                  delta_color="inverse" if adverse_events > 5 else "normal")
    
    # Overall health score
    st.markdown("---")
    st.subheader("🎯 System Health Score")
    
    # Calculate health score (0-100)
    health_score = 100
    health_score -= (active_crises * 20)  # Each crisis reduces score by 20
    health_score -= max(0, (adverse_events - 3) * 5)  # Adverse events over 3 reduce score
    health_score = max(0, health_score)  # Floor at 0
    
    col_left, col_right = st.columns([2, 1])
    
    with col_left:
        st.progress(health_score / 100, text=f"Health Score: {health_score}/100")
        
        if health_score >= 80:
            st.success("🟢 **Excellent** - No significant issues detected")
        elif health_score >= 60:
            st.info("🔵 **Good** - Minor monitoring required")
        elif health_score >= 40:
            st.warning("🟡 **Concerning** - Active monitoring required")
        else:
            st.error("🔴 **Critical** - Immediate action required!")
    
    with col_right:
        # Sentiment distribution pie chart
        sentiment_dist = session.sql("""
            SELECT 
                sentiment_category,
                COUNT(*) as count
            FROM post_sentiment_analysis
            WHERE analyzed_at >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
            GROUP BY sentiment_category
        """).to_pandas()
        
        if not sentiment_dist.empty:
            pie_chart = alt.Chart(sentiment_dist).mark_arc(innerRadius=50).encode(
                theta=alt.Theta(field="COUNT", type="quantitative"),
                color=alt.Color(field="SENTIMENT_CATEGORY", type="nominal",
                                scale=alt.Scale(
                                    domain=['POSITIVE', 'NEUTRAL', 'NEGATIVE'],
                                    range=['#00CC96', '#FFA15A', '#EF553B']
                                ),
                                legend=alt.Legend(title="Sentiment")),
                tooltip=['SENTIMENT_CATEGORY', 'COUNT']
            ).properties(
                width=250,
                height=250,
                title='Sentiment Distribution'
            )
            
            st.altair_chart(pie_chart, use_container_width=False)
    
    # Platform breakdown
    st.markdown("---")
    st.subheader("📱 Activity by Platform")
    
    platform_data = session.sql("""
        SELECT 
            p.platform,
            COUNT(*) as post_count,
            ROUND(AVG(s.sentiment_score), 3) as avg_sentiment
        FROM social_media_posts p
        INNER JOIN post_sentiment_analysis s ON p.post_id = s.post_id
        WHERE p.posted_timestamp >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
        GROUP BY p.platform
        ORDER BY post_count DESC
    """).to_pandas()
    
    if not platform_data.empty:
        col_left, col_right = st.columns(2)
        
        with col_left:
            # Post volume by platform
            bar_chart = alt.Chart(platform_data).mark_bar().encode(
                x=alt.X('POST_COUNT:Q', title='Post Count'),
                y=alt.Y('PLATFORM:N', title='Platform', sort='-x'),
                color=alt.Color('POST_COUNT:Q', scale=alt.Scale(scheme='blues'), legend=None),
                tooltip=['PLATFORM:N', 'POST_COUNT:Q', 'AVG_SENTIMENT:Q']
            ).properties(
                width=350,
                height=250,
                title='Post Volume by Platform'
            )
            
            st.altair_chart(bar_chart, use_container_width=True)
        
        with col_right:
            # Sentiment by platform
            sentiment_chart = alt.Chart(platform_data).mark_bar().encode(
                x=alt.X('AVG_SENTIMENT:Q', title='Avg Sentiment', scale=alt.Scale(domain=[-1, 1])),
                y=alt.Y('PLATFORM:N', title='Platform', sort='-x'),
                color=alt.condition(
                    alt.datum.AVG_SENTIMENT > 0,
                    alt.value('#00CC96'),  # Green for positive
                    alt.value('#EF553B')   # Red for negative
                ),
                tooltip=['PLATFORM:N', 'AVG_SENTIMENT:Q', 'POST_COUNT:Q']
            ).properties(
                width=350,
                height=250,
                title='Average Sentiment by Platform'
            )
            
            zero_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='gray').encode(x='x:Q')
            
            st.altair_chart(sentiment_chart + zero_line, use_container_width=True)

# ============================================================================
# TAB 2: ACTIVE CRISES
# ============================================================================
with tab2:
    st.subheader("🚨 Active Crisis Alerts")
    
    if active_crises > 0:
        crisis_data = session.sql("""
            SELECT 
                product,
                platform,
                alert_severity,
                current_sentiment,
                sentiment_z_score,
                adverse_event_count,
                volume_spike_pct,
                current_post_volume
            FROM crisis_detection_realtime
            WHERE status = 'CRISIS_DETECTED'
            ORDER BY 
                CASE alert_severity 
                    WHEN 'CRITICAL' THEN 1 
                    WHEN 'HIGH' THEN 2 
                    WHEN 'MEDIUM' THEN 3 
                    ELSE 4 
                END,
                sentiment_z_score
        """).to_pandas()
        
        # Display each crisis as an alert box
        for _, crisis in crisis_data.iterrows():
            severity = crisis['ALERT_SEVERITY']
            
            if severity == 'CRITICAL':
                st.error(f"""
                ### 🚨 CRITICAL ALERT - {crisis['PRODUCT']} on {crisis['PLATFORM']}
                
                **Sentiment Z-Score**: {crisis['SENTIMENT_Z_SCORE']:.2f}  
                **Current Sentiment**: {crisis['CURRENT_SENTIMENT']:.3f}  
                **Adverse Events**: {crisis['ADVERSE_EVENT_COUNT']}  
                **Volume Spike**: {crisis['VOLUME_SPIKE_PCT']:.1f}%  
                **Current Volume**: {crisis['CURRENT_POST_VOLUME']} posts
                """)
            elif severity == 'HIGH':
                st.warning(f"""
                ### ⚠️ HIGH PRIORITY - {crisis['PRODUCT']} on {crisis['PLATFORM']}
                
                **Sentiment Z-Score**: {crisis['SENTIMENT_Z_SCORE']:.2f}  
                **Current Sentiment**: {crisis['CURRENT_SENTIMENT']:.3f}  
                **Adverse Events**: {crisis['ADVERSE_EVENT_COUNT']}  
                **Volume Spike**: {crisis['VOLUME_SPIKE_PCT']:.1f}%  
                **Current Volume**: {crisis['CURRENT_POST_VOLUME']} posts
                """)
            else:
                st.info(f"""
                ### 🔵 MEDIUM PRIORITY - {crisis['PRODUCT']} on {crisis['PLATFORM']}
                
                **Sentiment Z-Score**: {crisis['SENTIMENT_Z_SCORE']:.2f}  
                **Current Sentiment**: {crisis['CURRENT_SENTIMENT']:.3f}  
                **Adverse Events**: {crisis['ADVERSE_EVENT_COUNT']}  
                **Volume Spike**: {crisis['VOLUME_SPIKE_PCT']:.1f}%  
                **Current Volume**: {crisis['CURRENT_POST_VOLUME']} posts
                """)
        
        # Crisis data table
        st.markdown("---")
        st.markdown("**📋 Crisis Details Table**")
        st.dataframe(
            crisis_data,
            use_container_width=True,
            hide_index=True,
            column_config={
                "ALERT_SEVERITY": st.column_config.TextColumn("Severity", width="small"),
                "CURRENT_SENTIMENT": st.column_config.NumberColumn("Sentiment", format="%.3f"),
                "SENTIMENT_Z_SCORE": st.column_config.NumberColumn("Z-Score", format="%.2f"),
            }
        )
    else:
        st.success("✅ **No active crises detected**")
        st.info("""All products and platforms are operating within normal parameters.  
                Continue monitoring for any changes in sentiment or volume.""")
    
    # Adverse events requiring review
    st.markdown("---")
    st.subheader("⚠️ Adverse Events Requiring Review")
    
    adverse_posts = session.sql("""
        SELECT 
            p.product_mentioned,
            p.platform,
            p.post_text,
            s.adverse_event_type,
            s.severity,
            s.sentiment_score,
            p.posted_timestamp
        FROM social_media_posts p
        INNER JOIN post_sentiment_analysis s ON p.post_id = s.post_id
        WHERE s.adverse_event_detected = TRUE
          AND p.posted_timestamp >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
        ORDER BY 
            CASE s.severity 
                WHEN 'critical' THEN 1 
                WHEN 'high' THEN 2 
                WHEN 'moderate' THEN 3 
                ELSE 4 
            END,
            p.posted_timestamp DESC
        LIMIT 10
    """).to_pandas()
    
    if len(adverse_posts) > 0:
        st.dataframe(adverse_posts, use_container_width=True, hide_index=True)
        
        # Download button
        csv = adverse_posts.to_csv(index=False)
        st.download_button(
            label="📥 Download Adverse Events Report",
            data=csv,
            file_name="adverse_events_report.csv",
            mime="text/csv"
        )
    else:
        st.info("✅ No adverse events detected in last 24 hours")

# ============================================================================
# TAB 3: TRENDS
# ============================================================================
with tab3:
    st.subheader("📈 Sentiment Trends Over Time")
    
    # Get hourly sentiment trends
    trend_data = session.sql("""
        SELECT 
            DATE_TRUNC('hour', p.posted_timestamp) as hour,
            ROUND(AVG(s.sentiment_score), 3) as avg_sentiment,
            COUNT(*) as post_volume
        FROM social_media_posts p
        INNER JOIN post_sentiment_analysis s ON p.post_id = s.post_id
        WHERE p.posted_timestamp >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
        GROUP BY DATE_TRUNC('hour', p.posted_timestamp)
        ORDER BY hour
    """).to_pandas()
    
    if not trend_data.empty:
        # Sentiment line chart
        base = alt.Chart(trend_data).encode(
            x=alt.X('HOUR:T', title='Time (Hourly)', axis=alt.Axis(format='%H:%M'))
        )
        
        line = base.mark_line(point=True, strokeWidth=3, color='#636EFA').encode(
            y=alt.Y('AVG_SENTIMENT:Q', title='Sentiment Score', scale=alt.Scale(domain=[-1, 1])),
            tooltip=[
                alt.Tooltip('HOUR:T', title='Hour', format='%Y-%m-%d %H:%M'),
                alt.Tooltip('AVG_SENTIMENT:Q', title='Sentiment', format='.3f'),
                alt.Tooltip('POST_VOLUME:Q', title='Posts')
            ]
        )
        
        # Threshold lines
        positive_line = alt.Chart(pd.DataFrame({'y': [0.5]})).mark_rule(
            strokeDash=[5, 5], color='green', opacity=0.5
        ).encode(y='y:Q')
        
        negative_line = alt.Chart(pd.DataFrame({'y': [-0.5]})).mark_rule(
            strokeDash=[5, 5], color='red', opacity=0.5
        ).encode(y='y:Q')
        
        zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
            strokeDash=[2, 2], color='gray', opacity=0.7
        ).encode(y='y:Q')
        
        final_chart = (line + positive_line + negative_line + zero_line).properties(
            width=800,
            height=400,
            title='24-Hour Sentiment Trend'
        )
        
        st.altair_chart(final_chart, use_container_width=True)
        
        # Trend statistics
        col1, col2, col3, col4 = st.columns(4)
        
        with col1:
            recent_6hr = trend_data.tail(6)['AVG_SENTIMENT'].mean()
            st.metric("Recent 6hr Avg", f"{recent_6hr:.3f}")
        
        with col2:
            peak = trend_data['AVG_SENTIMENT'].max()
            st.metric("Peak Sentiment", f"{peak:.3f}")
        
        with col3:
            valley = trend_data['AVG_SENTIMENT'].min()
            st.metric("Lowest Sentiment", f"{valley:.3f}")
        
        with col4:
            volatility = trend_data['AVG_SENTIMENT'].std()
            st.metric("Volatility (StdDev)", f"{volatility:.3f}")
    else:
        st.info("No trend data available yet")
    
    # Product comparison
    st.markdown("---")
    st.subheader("💊 Product Sentiment Comparison")
    
    product_sentiment = session.sql("""
        SELECT 
            p.product_mentioned as product,
            ROUND(AVG(s.sentiment_score), 3) as avg_sentiment,
            COUNT(*) as post_count,
            SUM(CASE WHEN s.adverse_event_detected THEN 1 ELSE 0 END) as adverse_count
        FROM social_media_posts p
        INNER JOIN post_sentiment_analysis s ON p.post_id = s.post_id
        WHERE p.posted_timestamp >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
        GROUP BY p.product_mentioned
        ORDER BY avg_sentiment DESC
    """).to_pandas()
    
    if not product_sentiment.empty:
        # Product sentiment bar chart
        product_chart = alt.Chart(product_sentiment).mark_bar().encode(
            x=alt.X('AVG_SENTIMENT:Q', title='Average Sentiment', scale=alt.Scale(domain=[-1, 1])),
            y=alt.Y('PRODUCT:N', title='Product', sort='-x'),
            color=alt.condition(
                alt.datum.AVG_SENTIMENT > 0,
                alt.value('#00CC96'),
                alt.value('#EF553B')
            ),
            tooltip=['PRODUCT:N', 'AVG_SENTIMENT:Q', 'POST_COUNT:Q', 'ADVERSE_COUNT:Q']
        ).properties(
            width=700,
            height=300,
            title='Average Sentiment by Product'
        )
        
        zero_line = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='gray').encode(x='x:Q')
        
        st.altair_chart(product_chart + zero_line, use_container_width=True)
        
        # Product details table
        st.markdown("**📋 Product Details**")
        st.dataframe(product_sentiment, use_container_width=True, hide_index=True)
    else:
        st.info("No product data available")

# ============================================================================
# TAB 4: DETAILS
# ============================================================================
with tab4:
    st.subheader("🔍 Detailed Post Analysis")
    
    # Filters
    col1, col2, col3 = st.columns(3)
    
    with col1:
        sentiment_filter = st.selectbox(
            "Filter by Sentiment",
            ["All", "Positive", "Neutral", "Negative"]
        )
    
    with col2:
        time_filter = st.selectbox(
            "Time Range",
            ["Last 6 hours", "Last 12 hours", "Last 24 hours"]
        )
    
    with col3:
        limit = st.slider("Number of posts", 10, 50, 20)
    
    # Build query based on filters
    if sentiment_filter == "Positive":
        where_sentiment = "AND s.sentiment_category = 'POSITIVE'"
    elif sentiment_filter == "Negative":
        where_sentiment = "AND s.sentiment_category = 'NEGATIVE'"
    elif sentiment_filter == "Neutral":
        where_sentiment = "AND s.sentiment_category = 'NEUTRAL'"
    else:
        where_sentiment = ""
    
    if time_filter == "Last 6 hours":
        hours = 6
    elif time_filter == "Last 12 hours":
        hours = 12
    else:
        hours = 24
    
    negative_posts = session.sql(f"""
        SELECT 
            p.product_mentioned as product,
            p.platform,
            p.post_text,
            s.sentiment_score,
            s.sentiment_category,
            p.engagement_count as engagement,
            p.posted_timestamp
        FROM social_media_posts p
        INNER JOIN post_sentiment_analysis s ON p.post_id = s.post_id
        WHERE p.posted_timestamp >= DATEADD('hour', -{hours}, CURRENT_TIMESTAMP())
        {where_sentiment}
        ORDER BY s.sentiment_score, p.engagement_count DESC
        LIMIT {limit}
    """).to_pandas()
    
    if len(negative_posts) > 0:
        st.dataframe(negative_posts, use_container_width=True, hide_index=True)
        
        # Download button
        csv = negative_posts.to_csv(index=False)
        st.download_button(
            label="📥 Download Post Data",
            data=csv,
            file_name=f"posts_{sentiment_filter.lower()}_{hours}hr.csv",
            mime="text/csv"
        )
    else:
        st.info("No posts match the selected filters")

# ============================================================================
# FOOTER
# ============================================================================
st.markdown("---")
col1, col2 = st.columns([3, 1])

with col1:
    st.success("✅ Crisis detection system operational | Monitoring active")

with col2:
    if st.button("🔄 Refresh Data"):
        st.rerun()

# Info section
with st.expander("ℹ️ About This Dashboard"):
    st.markdown("""
    ### Crisis Detection System
    
    **Overview Tab:**
    - System health score and metrics
    - Sentiment distribution
    - Platform activity breakdown
    
    **Active Crises Tab:**
    - Real-time crisis alerts by severity
    - Adverse event tracking
    - Downloadable reports
    
    **Trends Tab:**
    - 24-hour sentiment trends
    - Product comparison
    - Volatility analysis
    
    **Details Tab:**
    - Filterable post analysis
    - Custom time ranges
    - Export functionality
    
    **Crisis Detection Logic:**
    - **Critical**: Z-score < -3.0 or ≥5 adverse events
    - **High**: Z-score < -2.5 or ≥3 adverse events
    - **Medium**: Z-score < -2.0
    - **Low**: Minor anomalies
    """)

---

##  Tutorial Complete!

### What You've Learned

1.  **Real-time sentiment analysis** with `AI_SENTIMENT()`
2.  **LLM classification** with `AI_COMPLETE()`
3.  **Statistical baselines** using `AVG()` and `STDDEV()`
4.  **Anomaly detection** with z-score calculations
5.  **Crisis alerting** with severity classification
6.  **Interactive dashboards** with Streamlit

### Key SQL Techniques

- **Window functions** for rolling calculations
- **Statistical aggregations** for baselines
- **CASE statements** for classification logic
- **CTEs** for complex analysis pipelines
- **Views** for real-time monitoring

### Next Steps for Production

1. **Connect real data sources** (Twitter API, Meta API, Reddit API)
2. **Set up automated alerts** (email, Slack, PagerDuty)
3. **Customize thresholds** for your brand's baseline
4. **Add sentiment by topic** (efficacy, cost, availability)
5. **Integrate with pharmacovigilance** systems

---

**Congratulations!** You've built a production-ready social media crisis detection system that can monitor brand health 24/7 and alert your communications team to emerging threats in real-time.

**Estimated runtime**: (varies by data size)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `SOCIAL_MEDIA_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS SOCIAL_MEDIA_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
